In [ ]:
# ---------------------------------------------------------
# UNIFICACIÓN SEGURA: MANTENIENDO TODO EL HISTÓRICO
# ---------------------------------------------------------
print("\n[INFO] Unificando sin perder datos antiguos...")

# 1. Hacemos el merge manteniendo TODO el SENAMHI (how='left')
dataset_final = pd.merge(senamhi, era5_clean, how='left', on=['fecha', 'lat_grid', 'lon_grid'])

# 2. LIMPIEZA INTELIGENTE: Rellenar huecos de ERA5 con el promedio histórico
# Si ERA5 no tiene dato, ponemos el promedio de toda la columna para esa variable
columnas_clima = ['temp_2m_era5', 'precip_era5', 'presion_era5', 'dew_point_era5']
for col in columnas_clima:
    dataset_final[col] = dataset_final[col].fillna(dataset_final[col].mean())

# 3. Limpieza de columnas basura
dataset_final = dataset_final.drop(columns=['lat_grid', 'lon_grid', 'number'], errors='ignore')

# 4. Verificación
print(f"\n✅ UNIFICACIÓN COMPLETA")
print(f"Total de registros conservados: {dataset_final.shape[0]} (¡Ya no perdiste los del 2002!)")
print(f"¿Cuántos datos vacíos quedan?: {dataset_final.isnull().sum().sum()}")

# Guardar
dataset_final.to_csv('data_process/dataset_ML_final_completo.csv', index=False)
dataset_final.head()


[INFO] Unificando sin perder datos antiguos...

✅ UNIFICACIÓN COMPLETA
Total de registros conservados: 392281 (¡Ya no perdiste los del 2002!)
¿Cuántos datos vacíos quedan?: 792053


,year,month,day,precip,tmax,tmin,estacion,lat,lon,zona,departamento,fecha,amp_termica,helada,latitude,longitude,temp_2m_era5,precip_era5,presion_era5,dew_point_era5
0,2002,8,1,0.0,9.5,-3.0,ANANEA,-14.68,-69.53,Norte,PUNO,2002-08-01,12.5,1,NaN,NaN,6.57412,0.001924,62853.851886,-0.867013
1,2002,8,2,0.0,9.0,-3.5,ANANEA,-14.68,-69.53,Norte,PUNO,2002-08-02,12.5,1,NaN,NaN,6.57412,0.001924,62853.851886,-0.867013
2,2002,8,3,1.0,10.0,-6.0,ANANEA,-14.68,-69.53,Norte,PUNO,2002-08-03,16.0,1,NaN,NaN,6.57412,0.001924,62853.851886,-0.867013
3,2002,8,4,0.0,11.0,-2.0,ANANEA,-14.68,-69.53,Norte,PUNO,2002-08-04,13.0,1,NaN,NaN,6.57412,0.001924,62853.851886,-0.867013
4,2002,8,5,1.5,8.5,-1.5,ANANEA,-14.68,-69.53,Norte,PUNO,2002-08-05,10.0,1,NaN,NaN,6.57412,0.001924,62853.851886,-0.867013


In [8]:
# 1. Asegurar que tenemos mes y año para imputar mejor
dataset_final['mes'] = dataset_final['fecha'].dt.month

# 2. Imputación por Mes: Rellenar cada NaN con el promedio de ese mes específico
# Así, si falta un dato de ERA5 en un agosto de 2005, se rellena con el promedio de los agostos de 2015-2023
columnas_clima = ['temp_2m_era5', 'precip_era5', 'presion_era5', 'dew_point_era5']

for col in columnas_clima:
    dataset_final[col] = dataset_final[col].fillna(dataset_final.groupby('mes')[col].transform('mean'))

# 3. Si aún queda algún hueco (casos extremos), rellenar con el promedio global
dataset_final[columnas_clima] = dataset_final[columnas_clima].fillna(dataset_final[columnas_clima].mean())

print(f"✅ ¡Limpieza final completada! Datos vacíos restantes: {dataset_final.isnull().sum().sum()}")

✅ ¡Limpieza final completada! Datos vacíos restantes: 792053


In [9]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Definir qué columnas usa el modelo (X) y qué queremos predecir (y)
features = ['tmax', 'tmin', 'amp_termica', 'temp_2m_era5', 'dew_point_era5', 'presion_era5']
X = dataset_final[features]
y = dataset_final['helada']

# 2. Dividir en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Entrenar el modelo XGBoost
model = XGBClassifier()
model.fit(X_train, y_train)

# 4. Probar qué tan bien predice
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

ModuleNotFoundError: No module named 'sklearn'